# Project: Neural Estate
### Datum: 2026-03-11
### Teamleden: Michal Kakol (24087068), Sem Ooms (23091789), Chaimae

# 1. Setup:

Write a PEP8-compliant Python script using an Object-Oriented Programming framework. Create a shared Config class containing: a random seed of 42, a validation split ratio of 0.20, an evaluation metric of 'accuracy', and a string-to-integer mapping dictionary for the 8 music genres. Build a DatasetManager class using Pandas to ingest train.csv. Implement a stratified random splitter that creates immutable training and validation index subsets so that all individual pipelines run on identical cross-validation data slices to ensure strict parity for downstream late-fusion ensembling.
Data directory for one random train video: C:\Users\mkako\Portables\Projects\music_classification_dl\data\Train\blues\blues.00001.wav (there is also folder for country, disco, hiphop, metal, pop, reggae, rock).
Data directory for one random test video: C:\Users\mkako\Portables\Projects\music_classification_dl\data\Test\test.00000.wav
test database: C:\Users\mkako\Portables\Projects\music_classification_dl\data\test.csv
train database: C:\Users\mkako\Portables\Projects\music_classification_dl\data\train.csv

In [ ]:
# code cell

# 2. Exploratory Data Analysis (EDA):

Create an object-oriented EDA class DataExplorer using PEP8 conventions.Task 1 & 2: Read train.csv and display playable audio cells (IPython.display.Audio) for one representative track per genre folder.Task 3: Ingest audio files via librosa.load. Convert raw sample lengths ($N$) into explicit time vectors using the recording's sampling frequency ($sfreq$) via the formula $t_i = \frac{i}{sfreq}$. Print out file durations and sample rates to verify uniform grids. Treat the raw sound stream as a long 1D NumPy array.Task 4: Clean the lyrics text column (lowercase, strip punctuation). Calculate text token lengths per song, display a boxplot showing length distributions by genre, and extract a frequency chart of the top 5 words per genre.Task 5: Implement frequency-domain analysis plots for the genres. Compute a Short-Time Fourier Transform (STFT) using librosa.stft to convert time-domain waveforms into complex frequency-domain spectrograms. Use librosa.amplitude_to_db to yield decibel scaling. Append a clear markdown block explaining that according to Fourier principles, any complex, noisy acoustic wave can be decomposed into steady constituent sinusoids ($\mathbf{y(t) = A \cdot \sin(2\pi ft + \phi)}$). Mention the Nyquist criteria ($\frac{sfreq}{2}$) as the physical barrier preventing sampling aliasing. Show how the Mel scale projects these frequencies to match human auditory perception by emphasizing low-frequency variations over high-frequency shifts.

In [ ]:
# code cell

# 3. Recurrent Model for Audio Files:

Build an audio sequence network using Keras and TensorFlow.Task 3 (Feature Engineering): Write an audio feature extractor class that converts 30-second audio files into log-scaled Mel-Spectrogram arrays (librosa.feature.melspectrogram followed by librosa.power_to_db). Pad or truncate these spectrogram arrays along the time axis to enforce a clean, uniform fixed-shape sequence matrix across all samples.Task 1 & 4 (Architecture & Justification): Formulate a Keras Sequential model. Structure the network using two stacked recurrent layers (GRU(64, return_sequences=True) followed by GRU(64, return_sequences=False)), a Dense hidden layer (32 units, ReLU, with Dropout(0.2)), and an 8-node Dense output layer with a Softmax activation function. Add a detailed markdown block explaining how the structural Gated Recurrent Unit architecture prevents the vanishing gradient errors of traditional RNN backpropagation loops (where long-term memory is erased by repeated fractional scalar multiplications like $0.5 \times 0.5 \times 0.5 \approx 0$). Explain that its Update and Reset gates efficiently determine how much structural historical data is preserved or discarded.Task 5 (Compilation & Training): Compile the architecture using sparse_categorical_crossentropy and the Adam optimizer. Crucially, initialize a ModelCheckpoint callback monitoring val_loss with save_best_only=True to save the optimal weight configuration and protect our small validation pool from overfitting. Train for 20 epochs using our predefined DatasetManager splits.

In [ ]:
# code cell

# 4. LSTM Model for Lyrics:

Implement a text processing pipeline using Keras sequence layers.
Task 1 & 4 (Preprocessing): Create a text preparation class. Initialize a Keras Tokenizer. Convert text strings into sequence arrays using texts_to_sequences. Standardize shape dimensions by applying pad_sequences to build a uniform temporal window for classification.
Task 3 (Architecture & Justification): Build a Keras Sequential model. Layer 1 must be an Embedding layer (input_dim=10000, output_dim=100, matching your padded max sequence length). Follow with a Bidirectional LSTM layer (64 units), a Dropout(0.3) layer for regularization, and an 8-node Dense output layer with Softmax activation.
Write a markdown block explaining the Many-to-One sequence typology used here (mapping an entire collection of lyric tokens down to a single categorical genre label). Differentiate this classification structure from Many-to-Many models (used for text generation tasks) and Encoder-Decoder arrangements (used for Sequence-to-Sequence tasks like Neural Machine Translation). Note that our word embeddings map nominal tokens into dense spatial vectors where mathematical closeness represents shared semantic contextual meaning, similar to Word2Vec models (CBOW and Skip-gram architectures).
Task 5 (Compilation): Compile using sparse_categorical_crossentropy and the Adam optimizer. Include a ModelCheckpoint(monitor='val_loss', save_best_only=True) callback. Train for 15 epochs on our text splits.

In [ ]:
# code cell

# 5. Transformer for Lyrics:

Using the Hugging Face transformers library, write an object-oriented script to fine-tune a text classification model.
Task 2 & 3: Instantiate distilbert-base-uncased via AutoModelForSequenceClassification with num_labels=8. Note in markdown that this functions as an Encoder-based model (reading sentences bidirectionally to capture meaning), contrasting it with Decoder-only systems (like GPT models using Masked Self-Attention for generative Causal Language Modeling) and Encoder-Decoder architectures (like BART for translation).
Task 4 (Explanation & Advanced Setup): In a markdown block, explain the fine-tuning training paradigm: it aims to adapt a general-purpose model to a specific down-stream task by updating the final classification Head while slightly adjusting the pre-trained backbone weights via backpropagation. Detail your data pipeline optimization choices:
Lazy Loading: Using the Hugging Face dataset structures (map()) to stream samples from disk, saving physical RAM.
Tokenizer Processing: Utilizing subword tokenization to encode strings, injecting special tracking markers ([CLS] at the sequence head for categorization context and [SEP] for boundaries).
Dynamic Padding: Implementing DataCollatorWithPadding to dynamically pad token dimensions up to the maximum length found within the current execution batch rather than filling to the global dataset maximum length, significantly reducing training overhead.
Implementation: Configure Hugging Face TrainingArguments setting load_best_model_at_end=True and metric_for_best_model='eval_loss'. Bind the collection using the high-level Trainer class passing the model, args, your tokenized data splits, your tokenizer, and your dynamic data_collator. Execute trainer.train(). Map the final classification output Logits through a Softmax function to calculate explicit validation probabilities.

In [ ]:
# code cell

# 6. Model of Choice (Late-Fusion Prediction Ensemble):

Design a Late-Fusion Multi-Modal Prediction Ensemble pipeline class using Python and NumPy.Task 1, 3, & 4 (Ensemble Logic): Instead of forcing different deep learning frameworks to combine inside a single layer-input architecture, implement a Late-Fusion decision-level model. This class will ingest the saved, optimal validation weight probability distributions generated independently from:The Audio GRU sequential model.The fine-tuned Lyric DistilBERT Transformer model.Write a markdown block justifying this multi-modal choice: explaining how combining independent networks at the decision level leverages distinct high-level abstractions (timbral/frequency space vs. semantic text context) while completely avoiding framework compilation errors.Implementation: Program the ensemble class to compute a weighted mathematical average of the predicted classification probabilities:$$\mathbf{P_{final} = w_{audio} \cdot P_{audio} + w_{text} \cdot P_{text}}$$Implement a clean loop to grid-search the optimal validation blending weights (e.g., $0.4 \cdot P_{audio} + 0.6 \cdot P_{text}$) to maximize overall joint validation accuracy. Plot a heatmap or grid tracking ensemble performance across different weight distributions.

In [ ]:
# code cell

# 7. Findings and Conclusion:

Write a clean inference loop to compute final ensemble predictions for the unlabeled test dataset.Load the unlabeled test assets, generating processed Log-Mel Spectrogram arrays for the audio model, and tokenizer encodings for the DistilBERT model.Run standalone test predictions from both optimal checkpointed backbones to obtain separate probability arrays ($P_{audio}$ and $P_{text}$).Blend the test probability arrays using the optimized blending weights determined during your Phase 6 validation grid search.Use argmax(axis=-1) on the blended probabilities to find the final target index, and convert them back to original string genre labels using your global Config mapping.Export the columns to a file named submission.csv containing exactly two columns: filename and genre.Generate a clear markdown summary table comparing validation results across all configurations: Audio GRU, Text LSTM, Lyric DistilBERT, and the combined Late-Fusion Multi-Modal Ensemble.

In [ ]:
# code cell